# Lecture 15 — Normalizing Flows

**PHYG004 / PHY5006, 2026 Spring · Sogang University**
Prof. Young Woo Choi

---

## What you'll learn today

By the end of this notebook you will be able to:

1. Explain in plain words what a "normalizing flow" is and why it might be useful in physics.
2. Watch a Gaussian get deformed into a complicated 2D shape — and write down what's happening mathematically.
3. Build a small flow in JAX (≈30 lines of code) and train it on two-dimensional toy data.
4. Use the same flow to **sample a double-well potential** in one shot — no MCMC, no waiting.

> **Runtime tip.** Everything runs on the free Colab CPU in about 3 minutes. No GPU needed.

> **What you should already know.** From earlier lectures: how to train a small neural network in JAX with `flax.nnx`, what a probability density is, and what a Gaussian is. That's it. We don't assume you remember Jensen's inequality.

<!-- lecture15-visual:start:title-flow -->
<!-- lecture15-visual:end:title-flow -->


## 0. Setup

In [ ]:
# !pip install -q jax jaxlib optax flax matplotlib

import jax
import jax.numpy as jnp
import jax.random as jr
import optax
import flax.nnx as nnx
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from functools import partial

print(f"JAX backend: {jax.default_backend()}")


## 1. The problem in one sentence

> *Given a bunch of data points $x_1, x_2, \ldots, x_N$, learn how to make **new ones** that look just like them.*

That's what a **generative model** does. Concretely:

- We have photos of cats → we want to make new cat photos.
- We have molecular conformations from MD → we want to draw new ones cheaply.
- We have lattice gauge field configurations → we want a fast sampler.

Mathematically, we want a probability density $p_\theta(x)$ such that

$$
p_\theta(x) \;\approx\; p_{\text{data}}(x).
$$

Once we have it, two things become possible:
- **Sample**: draw a fresh $x \sim p_\theta$.
- **Evaluate**: ask "how plausible is this particular $x$?" by computing $p_\theta(x)$.

We've seen one way to do this — the VAE from Lecture 14. Today we'll see another way that has one big advantage: it gives **exact** densities, not approximations.


## 2. The big idea — deform a Gaussian

Here's the central trick. We **already know** how to sample a complicated distribution: the Gaussian.

```
z ~ N(0, I)        ← we can do this in one line of code
```

Suppose we have a smooth, **invertible** function $f$ that maps the Gaussian world to the data world:

$$
x \;=\; f(z), \qquad z \sim \mathcal{N}(0, I).
$$

Then sampling $x$ is just: draw $z$, push it through $f$. **One forward pass. Independent samples. No Markov chain.**

The whole game is: **learn $f$**.

Think of it as squishing and stretching a blob of clay. The Gaussian is a perfectly round blob. The data is some weird shape. A flow learns how to deform one into the other.


In [ ]:
# A picture is worth a paragraph. Let's watch a Gaussian get deformed.
key = jr.PRNGKey(0)
z = jr.normal(key, (2000, 2))  # 2000 samples from N(0, I)

# Three example deformations. We just hand-pick the functions for illustration.
def deform_A(z):  # mild stretch along x-axis
    return jnp.stack([z[:, 0] * 2.0, z[:, 1]], axis=1)

def deform_B(z):  # shear
    return jnp.stack([z[:, 0] + 0.5 * z[:, 1], z[:, 1]], axis=1)

def deform_C(z):  # nonlinear: bend into a "U"
    return jnp.stack([z[:, 0], z[:, 1] + 0.6 * z[:, 0]**2 - 1.0], axis=1)

fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
labels = ["Start: Gaussian", "Stretch $x$", "Shear", "Bend (nonlinear)"]
points = [z, deform_A(z), deform_B(z), deform_C(z)]

for ax, lbl, p in zip(axes, labels, points):
    p = np.asarray(p)
    ax.scatter(p[:, 0], p[:, 1], s=3, alpha=0.4, c="C0")
    ax.set_title(lbl); ax.set_aspect("equal")
    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)
    ax.grid(alpha=0.3)
plt.suptitle("Same 2000 Gaussian samples — pushed through different deformations $f$",
             fontsize=12)
plt.tight_layout(); plt.show()


**Take-away.** Even simple hand-picked deformations of a Gaussian already produce non-Gaussian shapes. A neural network gives us a **learnable** $f$ — we'll just choose its weights so that the output matches our data.

<!-- lecture15-visual:start:change-of-variables -->
<!-- lecture15-visual:end:change-of-variables -->


## 3. Warm-up in 1D — the "stretching factor"

We have a Gaussian source $z$ and a transformed variable $x = f(z)$. **What does the density of $x$ look like?**

Intuition: think of $z$ and $x$ as positions of grains of sand.

- Where $f$ **stretches** space (a big interval of $z$ gets mapped to a big interval of $x$), the sand gets **spread out** → density goes **down**.
- Where $f$ **squeezes** space (a big interval of $z$ collapses into a small interval of $x$), the sand gets **piled up** → density goes **up**.

The "amount of stretch" at a point is $\left|\dfrac{df}{dz}\right|$ — the local **slope**. Big slope means big stretch.

Conservation of probability gives the formula we'll use throughout:

$$
\boxed{\; p_X(x) \;=\; \frac{p_Z(z)}{\left|\dfrac{df}{dz}\right|}, \qquad x = f(z). \;}
$$

That's it. That's the entire mathematical content of normalizing flows in 1D. Let's see it in action.


In [ ]:
# Take a Gaussian z and stretch/squeeze it through f(z) = z + 0.6*sin(z).
# Then plot the density of x and compare with what the formula predicts.
N = 100_000
z = jr.normal(jr.PRNGKey(0), (N,))

def f(z):    return z + 0.6 * jnp.sin(z)
def fp(z):   return 1.0 + 0.6 * jnp.cos(z)   # df/dz, always positive -> f is invertible

x = f(z)

# Predicted density at any x: p_X(x) = p_Z(z) / |f'(z)| where z = f^{-1}(x).
# Easy version: parametrise by z, then x = f(z) and density follows.
z_grid = jnp.linspace(-4, 4, 1000)
x_grid = f(z_grid)
pz_grid = jnp.exp(-z_grid**2 / 2) / jnp.sqrt(2*jnp.pi)
px_predicted = pz_grid / jnp.abs(fp(z_grid))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].hist(np.asarray(z), bins=80, density=True, color="C0", alpha=0.6, label="$z$ samples")
axes[0].plot(z_grid, pz_grid, "k-", lw=2, label=r"$\mathcal{N}(0,1)$")
axes[0].set_title("Source: Gaussian"); axes[0].legend(); axes[0].set_xlabel("$z$")

axes[1].hist(np.asarray(x), bins=80, density=True, color="C1", alpha=0.6, label="$x = f(z)$ samples")
axes[1].plot(x_grid, px_predicted, "k-", lw=2, label="predicted $p_X(x)$")
axes[1].set_title("After $f(z) = z + 0.6\\sin(z)$"); axes[1].legend(); axes[1].set_xlabel("$x$")

for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


**Watch carefully.** The histogram (orange) matches the predicted density (black line). The shape is bumpy because where $\cos z$ is small the slope is small, the space is squeezed, and probability piles up.

Now everything in this notebook is going to be a higher-dimensional version of this picture, and a learnable version of this $f$.


## 4. Higher dimensions — and the one technical wrinkle

In $d$ dimensions the slope $|df/dz|$ becomes the **Jacobian determinant** $|\det J_f|$, which measures how a small volume element gets resized.

$$
p_X(x) \;=\; \frac{p_Z(z)}{|\det J_f(z)|}, \qquad x = f(z).
$$

This is the same formula you've used in physics whenever you change coordinates — for example $dx\,dy = r\,dr\,d\theta$, where the Jacobian factor is $r$.

For most general neural networks $f$, two things go wrong:

1. The network might **not be invertible** at all — we couldn't undo $x = f(z)$.
2. Computing $\det J_f$ costs $O(d^3)$. For an image with $d \sim 10^4$ pixels, that's billions of operations. Hopeless.

> The whole architecture design of normalizing flows is one long answer to: **how do we build an invertible network whose Jacobian determinant is cheap to compute?**

We'll see one famous answer next: **coupling layers** (Dinh, Sohl-Dickstein & Bengio 2016, "RealNVP").


## 5. The coupling-layer trick

The trick is almost embarrassingly simple. Split the dimensions of $z$ into two groups, say $z = (z_A, z_B)$. Then build a one-layer flow as:

$$
x_A = z_A \qquad\text{(leave half the dimensions alone)}
$$

$$
x_B = z_B \cdot e^{\,s(z_A)} \;+\; t(z_A) \qquad\text{(scale and shift the other half, based on the first half)}
$$

Here $s(\cdot)$ and $t(\cdot)$ are **any** neural networks. They produce a scaling factor $e^{s}$ and a shift $t$ for each transformed dimension.

### Why this works

**Invertibility.** Given $x_A, x_B$, recover the inputs by:

- $z_A = x_A$ (we kept it!)
- $z_B = (x_B - t(x_A)) \cdot e^{-s(x_A)}$

No matrix inverse, no iterative solver. We literally just kept half the input and stored enough information to undo the rest.

**Tractable Jacobian.** The Jacobian matrix of this map looks like

$$
J \;=\;
\begin{pmatrix}
I & 0 \\
\star & \mathrm{diag}\,e^{s(z_A)}
\end{pmatrix}
$$

which is **triangular**. The determinant of a triangular matrix is just the product of its diagonal:

$$
\det J \;=\; \prod_{i \in B} e^{\,s_i(z_A)} \;=\; e^{\sum_i s_i(z_A)}.
$$

So $\log|\det J|$ is just a sum — $O(d)$, not $O(d^3)$. Done.

### One layer isn't enough

A single coupling layer leaves half the dimensions untouched — so it's not very expressive. The fix: **stack many** coupling layers, and **alternate** which group is kept fixed:

- Layer 1: keep $A$ fixed, transform $B$.
- Layer 2: keep $B$ fixed, transform $A$.
- Layer 3: keep $A$ fixed, transform $B$.
- …

After half-a-dozen layers every dimension has been transformed conditioned on every other dimension, and the flow can model arbitrarily complicated shapes.

<!-- lecture15-visual:start:coupling-layer -->
<!-- lecture15-visual:end:coupling-layer -->


In [ ]:
# Cartoon diagram of one coupling layer.
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.set_xlim(0, 10); ax.set_ylim(0, 3.6); ax.axis("off")

def box(x, y, w, h, label, fc, ec="#37474f"):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.06",
                                 fc=fc, ec=ec, lw=1.2))
    ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=11)

def arr(x1, y1, x2, y2, color="black"):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=1.3))

box(0.2, 2.4, 1.0, 0.9, r"$z_A$", "#e3f2fd")          # input z_A (kept)
box(0.2, 0.8, 1.0, 0.9, r"$z_B$", "#fce4ec")          # input z_B (transformed)
box(3.0, 1.9, 1.6, 1.0, r"$s, t$" + "\n(neural net)", "#fff3e0")
box(5.8, 0.8, 2.2, 0.9, r"$z_B \cdot e^{s} + t$", "#fce4ec", ec="#c2185b")
box(8.8, 2.4, 1.0, 0.9, r"$x_A$", "#e3f2fd")
box(8.8, 0.8, 1.0, 0.9, r"$x_B$", "#fce4ec")

arr(1.2, 2.85, 3.0, 2.55)   # z_A -> net
arr(1.2, 2.85, 8.8, 2.85)   # z_A -> x_A (kept)
arr(4.6, 2.20, 5.8, 1.45)   # net -> scale&shift
arr(1.2, 1.25, 5.8, 1.25)   # z_B -> scale&shift
arr(8.0, 1.25, 8.8, 1.25)   # -> x_B

ax.text(5.0, 3.2, "kept identical", fontsize=10, color="#1565c0", ha="center")
ax.text(5.0, 0.25, r"reversible:  $z_B = (x_B - t)\,e^{-s}$",
        fontsize=11, ha="center", color="#c2185b")
plt.tight_layout(); plt.show()


## 6. Hands-on A — code one coupling layer

We're going to work in 2D ($d = 2$), so each group is a single coordinate. Each coupling layer is a small MLP $\to (s, t)$ followed by the scale-and-shift step.

Read the comments carefully — the code is the math.


In [ ]:
class AffineCoupling(nnx.Module):
    # mask[i] == 1  ->  dimension i is "kept"
    # mask[i] == 0  ->  dimension i is "transformed"
    def __init__(self, d, hidden, mask, *, rngs):
        self.mask = mask
        # The s and t networks are an ordinary MLP. They look at the kept dims
        # and produce a scaling and a shift for the transformed dims.
        self.net = nnx.Sequential(
            nnx.Linear(d, hidden, rngs=rngs), nnx.relu,
            nnx.Linear(hidden, hidden, rngs=rngs), nnx.relu,
            nnx.Linear(hidden, 2 * d, rngs=rngs),   # outputs s and t stacked
        )
        self.d = d

    def _scale_and_shift(self, kept):
        st = self.net(kept)
        s_raw, t = st[..., :self.d], st[..., self.d:]
        s = jnp.tanh(s_raw) * 2.0   # bound |s| < 2 for training stability
        return s, t

    def forward(self, z):
        # Input z, output x = (z_A, z_B * exp(s) + t).
        s, t = self._scale_and_shift(z * self.mask)
        transformed = 1.0 - self.mask
        x = z * jnp.exp(s * transformed) + t * transformed
        log_det = jnp.sum(s * transformed, axis=-1)
        return x, log_det

    def inverse(self, x):
        # Input x, output z = (x_A, (x_B - t) * exp(-s)).
        s, t = self._scale_and_shift(x * self.mask)
        transformed = 1.0 - self.mask
        z = (x - t * transformed) * jnp.exp(-s * transformed)
        log_det = -jnp.sum(s * transformed, axis=-1)
        return z, log_det


class RealNVP(nnx.Module):
    # A stack of coupling layers with alternating masks.
    def __init__(self, d=2, n_layers=8, hidden=64, *, rngs):
        masks = []
        for i in range(n_layers):
            mask = jnp.zeros(d)
            mask = mask.at[:d//2].set(1.0) if i % 2 == 0 else mask.at[d//2:].set(1.0)
            masks.append(mask)
        self.layers = nnx.List([AffineCoupling(d, hidden, m, rngs=rngs) for m in masks])
        self.d = d

    def forward(self, z):
        # Push z through every layer, accumulating log|det J|.
        log_det = jnp.zeros(z.shape[0]); x = z
        for layer in self.layers:
            x, ld = layer.forward(x); log_det += ld
        return x, log_det

    def inverse(self, x):
        log_det = jnp.zeros(x.shape[0]); z = x
        for layer in reversed(self.layers):
            z, ld = layer.inverse(z); log_det += ld
        return z, log_det

    def log_prob(self, x):
        # log p(x) = log p_Z(z) + log|det J_{f^{-1}}(x)|, with z = f^{-1}(x).
        z, log_det_inv = self.inverse(x)
        log_pz = -0.5 * jnp.sum(z**2, axis=-1) - 0.5 * self.d * jnp.log(2 * jnp.pi)
        return log_pz + log_det_inv

    def sample(self, key, n):
        z = jr.normal(key, (n, self.d))
        x, _ = self.forward(z)
        return x


In [ ]:
# Quick sanity check: forward then inverse should recover the input.
flow = RealNVP(d=2, n_layers=8, hidden=64, rngs=nnx.Rngs(0))
z_test = jr.normal(jr.PRNGKey(1), (200, 2))
x_test, _ = flow.forward(z_test)
z_back, _ = flow.inverse(x_test)
print(f"Max round-trip error: {float(jnp.max(jnp.abs(z_test - z_back))):.2e}")

n_params = sum(p.size for p in jax.tree.leaves(nnx.state(flow, nnx.Param)))
print(f"Parameters in the flow: {n_params:,}")


The round-trip error should be tiny — limited by floating-point precision, not by the network. The flow is *literally* invertible by construction.


## 7. Hands-on B — train on a 2D shape

We'll fit two toy datasets: **two moons** and **two concentric rings**. Neither can be captured by a single Gaussian, so they're a good test of the flow.

The training objective is *maximum likelihood*: maximise the average $\log p_\theta(x)$ on data.

In code: `loss = -flow.log_prob(x).mean()`. That's it.


In [ ]:
def make_moons(n, noise=0.06, key=jr.PRNGKey(0)):
    k1, k2, k3 = jr.split(key, 3)
    n_half = n // 2
    theta_upper = jnp.linspace(0, jnp.pi, n_half)
    theta_lower = jnp.linspace(0, jnp.pi, n - n_half)
    upper = jnp.stack([jnp.cos(theta_upper), jnp.sin(theta_upper)], axis=1)
    lower = jnp.stack([1 - jnp.cos(theta_lower), 1 - jnp.sin(theta_lower) - 0.5], axis=1)
    data = jnp.concatenate([upper, lower], axis=0)
    data += noise * jr.normal(k2, data.shape)
    return data[jr.permutation(k3, len(data))]


def make_rings(n, key=jr.PRNGKey(0)):
    k1, k2, k3 = jr.split(key, 3)
    n1, n2 = n // 2, n - n // 2
    theta1 = jr.uniform(k1, (n1,), maxval=2*jnp.pi)
    theta2 = jr.uniform(k2, (n2,), maxval=2*jnp.pi)
    r1 = 1.0 + 0.08 * jr.normal(k2, (n1,))
    r2 = 2.0 + 0.08 * jr.normal(k3, (n2,))
    return jnp.concatenate([
        jnp.stack([r1*jnp.cos(theta1), r1*jnp.sin(theta1)], axis=1),
        jnp.stack([r2*jnp.cos(theta2), r2*jnp.sin(theta2)], axis=1),
    ], axis=0)


# Visualise the targets
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
moons = make_moons(2000); rings = make_rings(2000)
axes[0].scatter(moons[:, 0], moons[:, 1], s=4, alpha=0.5); axes[0].set_title("two moons")
axes[1].scatter(rings[:, 0], rings[:, 1], s=4, alpha=0.5); axes[1].set_title("two rings")
for ax in axes: ax.set_aspect("equal"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
def train_flow(flow, data, n_epochs=300, batch_size=256, lr=3e-4, seed=42):
    optimizer = nnx.Optimizer(flow, optax.adam(lr), wrt=nnx.Param)

    def loss_fn(flow, x):
        return -flow.log_prob(x).mean()       # negative log-likelihood

    @nnx.jit
    def train_step(flow, optimizer, x):
        loss, grads = nnx.value_and_grad(loss_fn, argnums=nnx.DiffState(0, nnx.Param))(flow, x)
        optimizer.update(flow, grads)
        return loss

    losses = []
    key = jr.PRNGKey(seed)
    for epoch in range(n_epochs):
        key, sub = jr.split(key)
        perm = jr.permutation(sub, len(data))
        data_shuf = data[perm]
        ep_loss, nb = 0.0, 0
        for i in range(0, len(data), batch_size):
            xb = data_shuf[i:i+batch_size]
            if len(xb) < 2: continue
            ep_loss += float(train_step(flow, optimizer, xb)); nb += 1
        losses.append(ep_loss / max(nb, 1))
        if (epoch + 1) % 100 == 0:
            print(f"epoch {epoch+1:4d}   NLL = {losses[-1]:.4f}")
    return losses


In [ ]:
# Train on two moons
data_moons = make_moons(5000, key=jr.PRNGKey(10))
flow_moons = RealNVP(d=2, n_layers=8, hidden=64, rngs=nnx.Rngs(0))
loss_moons = train_flow(flow_moons, data_moons, n_epochs=300)


In [ ]:
# Train on rings
data_rings = make_rings(5000, key=jr.PRNGKey(20))
flow_rings = RealNVP(d=2, n_layers=8, hidden=64, rngs=nnx.Rngs(1))
loss_rings = train_flow(flow_rings, data_rings, n_epochs=300)


## 8. Did it work? — three diagnostics

We inspect the trained flow in three ways.

1. **Samples**: draw $z \sim \mathcal{N}(0, I)$, push through the flow, plot the result. Does it look like the data?
2. **Density**: evaluate $p_\theta(x)$ on a grid. Does it light up where the data is?
3. **Latent**: push the *data* backwards through the flow. Does it land on a Gaussian-looking blob?

If all three checks pass, the flow has done its job.


In [ ]:
def plot_flow_results(flow, data, title, ax_samples, ax_density, ax_latent,
                       xlim=(-3, 3), ylim=(-3, 3)):
    samples = np.asarray(flow.sample(jr.PRNGKey(99), 3000))
    ax_samples.scatter(samples[:, 0], samples[:, 1], s=3, alpha=0.4, c="C1")
    ax_samples.scatter(data[:500, 0], data[:500, 1], s=3, alpha=0.3, c="C0", label="data")
    ax_samples.set_title(f"{title} — samples vs data"); ax_samples.legend(fontsize=8, markerscale=3)
    ax_samples.set_aspect("equal"); ax_samples.grid(alpha=0.3)

    nx = 100
    xg = jnp.linspace(xlim[0], xlim[1], nx); yg = jnp.linspace(ylim[0], ylim[1], nx)
    xx, yy = jnp.meshgrid(xg, yg)
    grid = jnp.stack([xx.ravel(), yy.ravel()], axis=-1)
    log_p = np.asarray(flow.log_prob(grid)).reshape(nx, nx)
    ax_density.contourf(np.asarray(xx), np.asarray(yy), np.exp(log_p), levels=30, cmap="viridis")
    ax_density.set_title(f"{title} — learned density"); ax_density.set_aspect("equal")

    z_data, _ = flow.inverse(jnp.asarray(data[:2000]))
    z_data = np.asarray(z_data)
    ax_latent.scatter(z_data[:, 0], z_data[:, 1], s=3, alpha=0.3, c="C2")
    theta = np.linspace(0, 2*np.pi, 100)
    for r in [1, 2]:
        ax_latent.plot(r*np.cos(theta), r*np.sin(theta), "k--", alpha=0.3, lw=0.8)
    ax_latent.set_title(f"{title} — data pushed back to latent space")
    ax_latent.set_aspect("equal"); ax_latent.set_xlim(-4, 4); ax_latent.set_ylim(-4, 4)
    ax_latent.grid(alpha=0.3)


fig, axes = plt.subplots(2, 3, figsize=(13, 8))
plot_flow_results(flow_moons, np.asarray(data_moons), "Moons",
                  axes[0,0], axes[0,1], axes[0,2], xlim=(-1.5, 2.5), ylim=(-1.0, 1.5))
plot_flow_results(flow_rings, np.asarray(data_rings), "Rings",
                  axes[1,0], axes[1,1], axes[1,2], xlim=(-3, 3), ylim=(-3, 3))
plt.tight_layout(); plt.show()


**What to look for.**

- Orange samples match the blue data — the flow generates plausible new examples.
- The density (middle column) should concentrate along the data shape. This is the **exact** $p_\theta(x)$, not a bound — so visible artifacts are model limitations, not estimator noise.
- The latent scatter (right column) should look closer to a Gaussian than the original data. The rings example is deliberately harder for this tiny RealNVP, so imperfect normalization is a useful diagnostic.


## 9. Watch the deformation happen — layer by layer

Let's look at how the flow actually builds up its transformation. We feed Gaussian noise in and snapshot the points after each coupling layer.


In [ ]:
key = jr.PRNGKey(42)
z0 = jr.normal(key, (1500, 2))

snapshots = [np.asarray(z0)]
h = z0
for layer in flow_moons.layers:
    h, _ = layer.forward(h)
    snapshots.append(np.asarray(h))

n_show = min(len(snapshots), 9)
idx = np.linspace(0, len(snapshots)-1, n_show, dtype=int)

fig, axes = plt.subplots(1, n_show, figsize=(2.8*n_show, 2.8))
for ax, i in zip(axes, idx):
    pts = snapshots[i]
    ax.scatter(pts[:, 0], pts[:, 1], s=2, alpha=0.3)
    if i == 0:                       ax.set_title(r"$z \sim \mathcal{N}(0,I)$", fontsize=10)
    elif i == len(snapshots)-1:      ax.set_title(r"$x = f(z)$", fontsize=10)
    else:                            ax.set_title(f"after layer {i}", fontsize=10)
    ax.set_aspect("equal"); ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)
    ax.grid(alpha=0.2); ax.tick_params(labelsize=7)
plt.suptitle("Gaussian → Two Moons: morphing one layer at a time", fontsize=13, y=1.02)
plt.tight_layout(); plt.show()


Each layer makes a small, simple move along one axis. **Stack a dozen and you get a highly nonlinear, expressive transformation.** That's the whole magic.


## 10. Why a physicist should care — sampling a Boltzmann distribution

Here's a problem from statistical mechanics. Suppose a 2D system has potential

$$
U(x_1, x_2) \;=\; (x_1^2 - 1)^2 + \tfrac{1}{2} x_2^2.
$$

This is a **double well**: two minima at $(\pm 1, 0)$ separated by a small barrier at the origin. At temperature $T$, the equilibrium distribution is the Boltzmann distribution

$$
p(x) \;\propto\; e^{-U(x) / k_B T}.
$$

We'd like to draw independent samples from $p(x)$. Standard tool: MCMC. But MCMC has a famous weakness — at low $T$ the chain **gets trapped** in one well for a very long time before it crosses the barrier.

A flow is a one-shot sampler: it doesn't take local steps, so it doesn't see the barrier.

<!-- lecture15-visual:start:boltzmann-generator -->
<!-- lecture15-visual:end:boltzmann-generator -->


In [ ]:
def double_well(x, kT=0.2):
    # Return U(x)/kT  (so we never have to divide later).
    return ((x[..., 0]**2 - 1)**2 + 0.5 * x[..., 1]**2) / kT

# Visualise the energy landscape and the Boltzmann density.
nx = 200
xg = jnp.linspace(-2, 2, nx); yg = jnp.linspace(-2, 2, nx)
xx, yy = jnp.meshgrid(xg, yg); grid = jnp.stack([xx, yy], axis=-1)
U_grid = np.asarray(double_well(grid, kT=1.0))    # U itself (kT=1 -> no rescale)
boltz = np.exp(-U_grid / 0.2); boltz /= boltz.sum() * (4.0/nx)**2

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im = axes[0].contourf(np.asarray(xx), np.asarray(yy), U_grid, levels=30, cmap="magma_r")
axes[0].set_title(r"Potential $U(x_1, x_2)$"); axes[0].plot([-1, 1], [0, 0], "w*", ms=12, mec="k")
plt.colorbar(im, ax=axes[0])
axes[1].contourf(np.asarray(xx), np.asarray(yy), boltz, levels=30, cmap="viridis")
axes[1].set_title(r"Boltzmann density at $k_BT = 0.2$"); axes[1].plot([-1, 1], [0, 0], "w*", ms=12, mec="k")
for ax in axes: ax.set_aspect("equal"); ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
plt.tight_layout(); plt.show()


### Training a flow on energy alone — no data needed

The neat thing is that we **don't have any data here**. We just have an energy function $U(x)$. Can we still train a flow?

Yes — by using a clever loss. We push Gaussian samples through the flow and ask: *does the output look Boltzmann-distributed under $U$?*

The loss turns out to be

$$
\mathcal{L}(\theta) \;=\; \mathbb{E}_{z \sim \mathcal{N}(0, I)}\!\left[\; \frac{U(f_\theta(z))}{k_B T} \;-\; \log|\det J_{f_\theta}(z)|\;\right].
$$

The first term says "low energy is good". The second says "spread your samples out". Their balance gives you the Boltzmann distribution.

(For the curious: this is the *reverse* KL divergence from the flow's density to the Boltzmann target. The derivation is in any textbook on variational inference, or in the appendix below.)


In [ ]:
kT = 0.2
flow_boltz = RealNVP(d=2, n_layers=12, hidden=64, rngs=nnx.Rngs(42))
optimizer_b = nnx.Optimizer(flow_boltz, optax.adam(3e-4), wrt=nnx.Param)

def boltzmann_loss(flow, z, kT):
    x, log_det = flow.forward(z)
    return (double_well(x, kT=kT) - log_det).mean()

@nnx.jit
def train_step_b(flow, optimizer, z, kT):
    loss, grads = nnx.value_and_grad(boltzmann_loss, argnums=nnx.DiffState(0, nnx.Param))(flow, z, kT)
    optimizer.update(flow, grads)
    return loss

losses_b, key = [], jr.PRNGKey(0)
for epoch in range(600):
    key, sub = jr.split(key)
    z = jr.normal(sub, (512, 2))
    losses_b.append(float(train_step_b(flow_boltz, optimizer_b, z, kT)))
    if (epoch+1) % 200 == 0:
        print(f"epoch {epoch+1:4d}   loss = {losses_b[-1]:.3f}")

plt.figure(figsize=(7, 3))
plt.plot(losses_b); plt.xlabel("epoch"); plt.ylabel("loss")
plt.title("Training a Boltzmann sampler from the energy alone"); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# Did it work? Compare flow samples to the exact Boltzmann density.
flow_samples = np.asarray(flow_boltz.sample(jr.PRNGKey(99), 5000))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].scatter(flow_samples[:, 0], flow_samples[:, 1], s=3, alpha=0.3, c="C1")
axes[0].set_title("Trained flow — 5000 one-shot samples"); axes[0].plot([-1, 1], [0, 0], "k*", ms=10)

nx = 100; xg = jnp.linspace(-2.2, 2.2, nx); yg = jnp.linspace(-2.2, 2.2, nx)
xx, yy = jnp.meshgrid(xg, yg); grid = jnp.stack([xx.ravel(), yy.ravel()], axis=-1)
log_q = np.asarray(flow_boltz.log_prob(grid)).reshape(nx, nx)
axes[1].contourf(np.asarray(xx), np.asarray(yy), np.exp(log_q), levels=30, cmap="viridis")
axes[1].set_title("Flow density $q_\\theta(x)$"); axes[1].plot([-1, 1], [0, 0], "w*", ms=10, mec="k")

U_grid = np.asarray(double_well(grid, kT=kT)).reshape(nx, nx)
exact = np.exp(-U_grid); exact /= exact.sum() * (4.4/nx)**2
axes[2].contourf(np.asarray(xx), np.asarray(yy), exact, levels=30, cmap="viridis")
axes[2].set_title("Exact Boltzmann $\\propto e^{-U/k_BT}$"); axes[2].plot([-1, 1], [0, 0], "w*", ms=10, mec="k")

for ax in axes:
    ax.set_xlim(-2.2, 2.2); ax.set_ylim(-2.2, 2.2); ax.set_aspect("equal")
    ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
plt.tight_layout(); plt.show()


**Look what happened.** The flow:

- visits **both wells** in roughly equal proportion — exactly what the symmetric Boltzmann distribution wants;
- reproduces the exact density (rightmost panel) closely;
- **needed no training data** — just the energy function.

This is the idea behind **Boltzmann generators** (Noé *et al.*, *Science* 2019). In production it's used for sampling protein conformations, drawing lattice gauge configurations, and so on.


## 11. Why we can't just use MCMC

Could we have sampled the same double well with a Metropolis chain? Let's see.

<!-- lecture15-visual:start:mcmc-vs-flow -->
<!-- lecture15-visual:end:mcmc-vs-flow -->


In [ ]:
# Simple Metropolis-Hastings, written with jax.lax.scan so it's fast.
@partial(jax.jit, static_argnames=("energy_fn", "n_steps", "step_size"))
def metropolis_mcmc(energy_fn, x0, kT, n_steps, step_size, key):
    def step(carry, k):
        x, n_acc = carry
        k1, k2 = jr.split(k)
        x_prop = x + step_size * jr.normal(k1, x.shape)
        dE = energy_fn(x_prop, kT) - energy_fn(x, kT)
        accept = jr.uniform(k2) < jnp.exp(-dE)
        x_new = jnp.where(accept, x_prop, x)
        return (x_new, n_acc + accept.astype(jnp.int32)), x_new
    keys = jr.split(key, n_steps)
    (_, n_acc), traj = jax.lax.scan(step, (x0, jnp.int32(0)), keys)
    return traj, n_acc

n_steps, step_size = 8000, 0.03
traj_L, _ = metropolis_mcmc(double_well, jnp.array([-1.0, 0.0]), kT, n_steps, step_size, jr.PRNGKey(0))
traj_R, _ = metropolis_mcmc(double_well, jnp.array([ 1.0, 0.0]), kT, n_steps, step_size, jr.PRNGKey(1))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].scatter(np.asarray(traj_L)[:, 0], np.asarray(traj_L)[:, 1], s=2, alpha=0.2, c="C3")
axes[0].set_title("MCMC — start in LEFT well")
axes[1].scatter(np.asarray(traj_R)[:, 0], np.asarray(traj_R)[:, 1], s=2, alpha=0.2, c="C4")
axes[1].set_title("MCMC — start in RIGHT well")
axes[2].scatter(flow_samples[:, 0], flow_samples[:, 1], s=2, alpha=0.2, c="C1")
axes[2].set_title("Flow — independent samples")
for ax in axes:
    ax.set_xlim(-2.2, 2.2); ax.set_ylim(-2.2, 2.2); ax.set_aspect("equal")
    ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$"); ax.plot([-1, 1], [0, 0], "k*", ms=10)
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


**Trapped on classroom timescales.** With small local proposals, each MCMC chain mostly explores the well where it started. The barrier height is $\Delta U / k_B T = 5$, so barrier-scale uphill proposals are exponentially suppressed by roughly $e^{-5}$.

A real-world barrier in chemistry can be $\Delta U / k_B T \sim 10$ to $30$, putting the crossing time far beyond any practical MCMC.

The flow avoids this local-walk bottleneck: it transforms the entire Gaussian distribution in one shot.


## 12. Recap

| What | In words |
|---|---|
| **Goal** | Learn an invertible $f$ that deforms a Gaussian into the data. |
| **Density** | $\log p_\theta(x) = \log p_Z(z) + \log|\det J_{f^{-1}}(x)|$ — exact, not a bound. |
| **Hard part** | Building $f$ so $\det J$ is cheap. |
| **Trick** | Coupling layers — split the variables, transform one half conditionally on the other. The Jacobian becomes triangular and the determinant collapses to a sum. |
| **Training (with data)** | Maximum likelihood = minimise $-\log p_\theta(x)$ on samples. |
| **Training (without data)** | Use the energy: $\mathcal{L} = \mathbb{E}_z[U(f(z))/k_BT - \log|\det J_f|]$. |
| **Killer app for physics** | Sampling Boltzmann distributions that MCMC cannot reach. |

### What's the catch?

- The latent and the data have the **same dimension**. A flow cannot compress like a VAE does.
- Designing a flow with both **expressiveness** and a **cheap Jacobian** is an active area of research. We saw the simplest (coupling). Others (autoregressive flows, neural splines, continuous flows) trade these off differently.
- Image-scale flows (Glow) work, but state-of-the-art image generation is now diffusion. We'll see diffusion in L16.


## 13. Going further (optional reading)

If you want to dig deeper after class, here are the keywords. We won't develop any of these in lecture.

- **More expressive flows** — Glow (1×1 invertible conv + actnorm), Neural Spline Flows (monotone splines instead of affine), Masked Autoregressive Flow (MAF), Inverse Autoregressive Flow (IAF).
- **Continuous flows** — define $dx/dt = v_\theta(x, t)$ instead of discrete layers. The Jacobian becomes a trace. This is the bridge to **diffusion models** (next week).
- **Equivariant flows** — bake physical symmetries (translation, rotation, periodic boundaries) directly into the architecture. Essential for serious molecular and lattice applications.
- **Boltzmann generators in production** — Noé *et al.* (*Science* 2019) for protein conformations; Albergo–Kanwar–Shanahan (*PRD* 2019) for lattice QCD.

For the math:
- The connection between maximum-likelihood training and the **forward KL** between data and model.
- The Boltzmann-generator loss is the **reverse KL** from model to target — and this direction has a famous failure mode (mode collapse) that production systems work around.

We covered just enough to make the picture honest. The slides under `slides/` walk through the variational-inference background (ELBO, reparameterization trick) if you want that.


## 14. References

**Foundational papers**

1. Dinh, Sohl-Dickstein & Bengio, *Density Estimation Using Real-Valued Non-Volume Preserving (Real-NVP) Transformations*, ICLR 2017. arXiv:1605.08803.
2. Kingma & Dhariwal, *Glow: Generative Flow with Invertible 1×1 Convolutions*, NeurIPS 2018. arXiv:1807.03039.
3. Rezende & Mohamed, *Variational Inference with Normalizing Flows*, ICML 2015. arXiv:1505.05770.

**Physics applications**

4. Noé, Olsson, Köhler & Wu, *Boltzmann Generators: Sampling Equilibrium States of Many-Body Systems with Deep Learning*, *Science* 365, eaaw1147 (2019).
5. Albergo, Kanwar & Shanahan, *Flow-based Generative Models for MCMC in Lattice Field Theory*, *Phys. Rev. D* 100, 034515 (2019).

**Reviews**

6. Papamakarios *et al.*, *Normalizing Flows for Probabilistic Modeling and Inference*, JMLR 22 (2021). arXiv:1912.02762.
7. Kobyzev, Prince & Brubaker, *Normalizing Flows: An Introduction and Review of Current Methods*, IEEE TPAMI 43, 3964 (2021).

---

*End of Lecture 15.*
